# Lektion 03 - Agentbaserade designmönster

I denna lektion utforskar vi tre grundläggande designmönster för att bygga effektiva AI-agenter:

1. **Tydliga agentinstruktioner** — Skapa precisa, rolldefinierande uppmaningar som styr agentens beteende
2. **Strukturerad utdata med Pydantic-modeller** — Säkerställa att agenter returnerar förutsägbar, validerad data
3. **Agenter med ett enda ansvar** — Designa fokuserade agenter som gör en sak riktigt bra

Vi kommer att tillämpa varje mönster på ett **scenario för rekommendation av resmål**, där vi steg för steg bygger ett system som kan föreslå resmål, kontrollera tillgänglighet och hantera logistik.


## Installation


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Mönster 1: Tydliga Agentinstruktioner

Det mest effektfulla mönstret är också det enklaste: att skriva tydliga, detaljerade instruktioner för din agent.

Bra instruktioner definierar:
- **Vem** agenten är (persona och ton)
- **Vad** den ska göra (steg-för-steg-ansvar)
- **Hur** den ska bete sig (begränsningar och stil)

Nedan skapar vi en reseconcierge-agent med tydliga instruktioner som formar varje svar den producerar.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Mönster 2: Strukturerad utdata med Pydantic-modeller

Fri formtext är användbar för konversation, men nedströmsystem behöver strukturerad data.
Genom att para ihop **Pydantic-modeller** med en **verktygsfunktion** kan vi:

- Definiera ett exakt schema för agentens utdata
- Validera svar automatiskt
- Integrera agentresultat i applikationslogik pålitligt

Nyckeln till efterlevnad är att skicka `response_format` när vi kör agenten. Detta tvingar
modellen att returnera ett validerat `TravelRecommendations`-objekt (tillgängligt på `response.value`)
istället för fri formtext. Verktyget `get_destination_details` returnerar också en typad
`DestinationRecommendation`, så datan förblir strukturerad från början till slut.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## Mönster 3: Agent med enskilt ansvar

Komplexa uppgifter gynnas av att dela upp arbetet mellan flera fokuserade agenter, var och en med ett enda ansvar:

- En **Destinationsexpert** som känner till platser och tillgänglighet
- En **Logistikplanerare** som hanterar flyg, hotell och resplaner

Detta speglar principen för *separation av ansvar* inom mjukvaruutveckling — varje agent är enklare att testa, underhålla och förbättra oberoende av varandra.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Sammanfattning

I denna lektion tillämpade vi tre agentbaserade designmönster på ett scenario för reseförslag:

| Mönster | Nyckelidé | Fördel |
|---|---|---|
| **Tydliga instruktioner** | Definiera persona, ansvar och begränsningar i förväg | Konsekvent agentbeteende som följer varumärket |
| **Strukturerat utdata** | Använd Pydantic-modeller som svarformat | Validerade, maskinläsbara resultat |
| **Enkel ansvarsfördelning** | Ge varje agent ett fokuserat jobb | Enklare att testa, underhålla och kombinera |

Dessa mönster fungerar naturligt tillsammans — du kan kombinera tydliga instruktioner med strukturerat utdata i en agent med enkel ansvarsfördelning för att bygga robusta, produktionsklara system.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, var vänlig notera att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess modersmål bör betraktas som den auktoritativa källan. För kritisk information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som uppstår till följd av användningen av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
